In [1]:
!pip install gensim
!pip install fasttext

  Using cached fasttext-0.9.3.tar.gz (73 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build fasttext


  error: subprocess-exited-with-error
  
  × Building wheel for fasttext (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [48 lines of output]
      C:\Users\jdorn\AppData\Local\Temp\pip-build-env-bacpozj9\overlay\Lib\site-packages\setuptools\dist.py:599: SetuptoolsDeprecationWarning: Invalid dash-separated key 'description-file' in 'metadata' (setup.cfg), please use the underscore name 'description_file' instead.
      !!
      
              ********************************************************************************
              Usage of dash-separated 'description-file' will not be supported in future
              versions. Please use the underscore name 'description_file' instead.
              (Affected: fasttext).
      
              Available configuration options are listed in:
              https://setuptools.pypa.io/en/latest/userguide/declarative_config.html
      
              This deprecation is overdue, please update your project and remove depre

# Importations

In [ ]:
import urllib.request
import zipfile
import fasttext
import os

In [ ]:
# Extract data

# Define the URL and the target filename
url = 'https://perso.limsi.fr/neveol/TP_ISD2020.zip'
zip_filename = 'TP_ISD2020.zip'

# Download the file from the link
urllib.request.urlretrieve(url, zip_filename)

# Extract the contents into a folder named 'extracted_data'
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('extracted_data')

print('Download and extraction complete!')

Download and extraction complete!


In [ ]:
# Create output directory
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
DATA_DIR = "extracted_data"
PRESS_DIR = f"{DATA_DIR}/QUAERO_FrenchPress"
MED_DIR = f"{DATA_DIR}/QUAERO_FrenchMed"

# FastTest Embedding (CBow)


In [ ]:
class FastText_Embedding():
  def __init__(self,
               emb_model="cbow",
               model_path=f"{OUTPUT_DIR}/fattest_embeddings.bin",
               load_model=False):
    self.emb_model = emb_model
    self.model_path = model_path
    if load_model and os.path.exists(self.model_path):
      self.model = fasttext.load_model(model_path)

  def fit(self, train_set: str = "QUAERO_FrenchMed_traindev.ospl"):
    self.model = fasttext.train_unsupervised(
        train_set,
        model=self.emb_model,
        dim=100,
        minCount=1,
    )
    self.model.save_model(self.model_path)
    return self

  def embed(self, word:str):
    # Predict the embedding of the word
    return self.model[word]

  def similar_words(self, word:str):
    return self.model.get_nearest_neighbors(word)



In [ ]:
Med_Emb = FastText_Embedding(model_path=f"{OUTPUT_DIR}/fasttext_med.bin").fit(
    train_set=f"{MED_DIR}/QUAERO_FrenchMed_traindev.ospl")
Press_Emb = FastText_Embedding(model_path=f"{OUTPUT_DIR}/fasttext_press.bin").fit(
    train_set=f"{PRESS_DIR}/QUAERO_FrenchPress_traindev.ospl")


In [ ]:
print("Med vocabulary size:", len(Med_Emb.model.get_words()))
print("Press vocabulary size:", len(Press_Emb.model.get_words()))

Med vocabulary size: 9105
Press vocabulary size: 39655


In [ ]:
print(len(Med_Emb.embed("rideaux"))) # should return a 100-vector
print(len(Press_Emb.embed("rideaux"))) # should return a 100-vector

100
100
